# Climate Disease Model

This notebook explores the relationship between climate variables and disease prevalence, using machine learning techniques.

In [2]:
%pip install pandas numpy scikit-learn matplotlib seaborn folium joblib plotly nbformat>=4.2.0 --upgrade --quiet

Note: you may need to restart the kernel to use updated packages.


In [3]:
# Import necessary libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score, classification_report
from sklearn.linear_model import LinearRegression, LogisticRegression

# Set display options
pd.set_option('display.max_columns', None)
sns.set(style="whitegrid")
%matplotlib inline

## 1. Data Loading

Load the climate and disease datasets from the data folder.

In [4]:
# Load climate data
climate_data = pd.read_csv('../data/climate_data.csv')

# Load disease data
disease_data = pd.read_csv('../data/disease_data.csv')

# Display the first few rows of each dataset
print("Climate Data:")
climate_data.head()

FileNotFoundError: [Errno 2] No such file or directory: '../data/climate_data.csv'

In [ ]:
print("Disease Data:")
disease_data.head()

## 2. Data Exploration and Analysis

In [ ]:
# Check basic information about the datasets
print("Climate Data Info:")
climate_data.info()
print("\nDisease Data Info:")
disease_data.info()

In [ ]:
# Statistical summary
print("Climate Data Summary:")
climate_data.describe()

print("\nDisease Data Summary:")
disease_data.describe()

In [ ]:
# Check for missing values
print("Missing values in climate data:")
print(climate_data.isnull().sum())
print("\nMissing values in disease data:")
print(disease_data.isnull().sum())

In [ ]:
# Merge datasets on common identifiers (assuming there's a common column like 'location_id' and 'date')
# Adjust the merge columns based on your actual data structure
merged_data = pd.merge(climate_data, disease_data, on=['location_id', 'date'], how='inner')
merged_data.head()

In [ ]:
# Visualize correlations between climate variables and disease incidence
plt.figure(figsize=(12, 10))
correlation_matrix = merged_data.corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt=".2f")
plt.title("Correlation Matrix of Climate and Disease Variables")
plt.show()

In [ ]:
# Plot disease incidence against key climate variables
climate_vars = ['temperature', 'humidity', 'rainfall']  # Adjust based on your actual column names
disease_var = 'disease_cases'  # Adjust based on your actual column name

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for i, var in enumerate(climate_vars):
    sns.scatterplot(data=merged_data, x=var, y=disease_var, ax=axes[i])
    axes[i].set_title(f'{var.capitalize()} vs {disease_var.replace("_", " ").capitalize()}')
plt.tight_layout()
plt.show()

## 3. Data Preprocessing (100%)

Comprehensive preprocessing of the climate and disease data.

In [ ]:
# Step 1: Handle missing values
print("Before handling missing values:")
print(merged_data.isnull().sum())

# Numeric columns - impute with median
numeric_features = merged_data.select_dtypes(include=['int64', 'float64']).columns.tolist()
# Remove the target variable from features if it's numeric
if disease_var in numeric_features:
    numeric_features.remove(disease_var)
    
# Categorical columns - impute with most frequent value
categorical_features = merged_data.select_dtypes(include=['object']).columns.tolist()

# Create preprocessing pipelines
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# Combine preprocessing steps
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

# Create a copy of the data for preprocessing
processed_data = merged_data.copy()

In [ ]:
# Step 2: Handle outliers using IQR method for numerical features
for column in numeric_features:
    Q1 = processed_data[column].quantile(0.25)
    Q3 = processed_data[column].quantile(0.75)
    IQR = Q3 - Q1
    
    # Define outliers boundaries
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    # Cap outliers
    processed_data[column] = np.where(processed_data[column] < lower_bound, lower_bound, processed_data[column])
    processed_data[column] = np.where(processed_data[column] > upper_bound, upper_bound, processed_data[column])

# Display summary after outlier treatment
processed_data[numeric_features].describe()

In [ ]:
# Step 3: Feature Engineering

# Create seasonal features if date is available
if 'date' in processed_data.columns:
    processed_data['date'] = pd.to_datetime(processed_data['date'])
    processed_data['month'] = processed_data['date'].dt.month
    processed_data['season'] = pd.cut(
        processed_data['month'], 
        bins=[0, 3, 6, 9, 12], 
        labels=['Winter', 'Spring', 'Summer', 'Fall'],
        include_lowest=True
    )
    
# Create interaction features between climate variables
if 'temperature' in processed_data.columns and 'humidity' in processed_data.columns:
    processed_data['temp_humidity_interaction'] = processed_data['temperature'] * processed_data['humidity']
    
if 'rainfall' in processed_data.columns and 'temperature' in processed_data.columns:
    processed_data['rain_temp_interaction'] = processed_data['rainfall'] * processed_data['temperature']

# Display new features
print("New features added:")
processed_data.head()

In [ ]:
# Step 4: Prepare features and target variables

# Update categorical features list if we added new categorical columns
categorical_features = processed_data.select_dtypes(include=['object']).columns.tolist()

# Define features and target
X = processed_data.drop([disease_var, 'date'], axis=1, errors='ignore')
y = processed_data[disease_var]

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training data shape: {X_train.shape}")
print(f"Testing data shape: {X_test.shape}")

## 4. Model Building

In [ ]:
# Update numeric and categorical features after feature engineering
numeric_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X.select_dtypes(include=['object']).columns.tolist()

# Rebuild preprocessor with updated features
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

# Determine if this is a regression or classification task
if len(np.unique(y)) > 10:  # If there are many unique values, assume regression
    model = RandomForestRegressor(n_estimators=100, random_state=42)
    evaluation_metric = "Mean Squared Error and R²"
else:  # If there are few unique values, assume classification
    model = RandomForestClassifier(n_estimators=100, random_state=42)
    evaluation_metric = "Accuracy and Classification Report"

# Create and fit pipeline
pipeline = Pipeline(steps=[('preprocessor', preprocessor), ('model', model)])
pipeline.fit(X_train, y_train)

# Make predictions
y_pred = pipeline.predict(X_test)

## 5. Model Evaluation

In [ ]:
# Evaluate the model
if len(np.unique(y)) > 10:  # Regression metrics
    mse = mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    print(f"Mean Squared Error: {mse:.4f}")
    print(f"R² Score: {r2:.4f}")
    
    # Visualize actual vs predicted values
    plt.figure(figsize=(10, 6))
    plt.scatter(y_test, y_pred, alpha=0.5)
    plt.plot([y.min(), y.max()], [y.min(), y.max()], 'k--', lw=2)
    plt.xlabel('Actual')
    plt.ylabel('Predicted')
    plt.title('Actual vs Predicted Disease Cases')
    plt.show()
else:  # Classification metrics
    accuracy = accuracy_score(y_test, y_pred)
    report = classification_report(y_test, y_pred)
    print(f"Accuracy: {accuracy:.4f}")
    print("Classification Report:")
    print(report)
    
    # Confusion matrix
    plt.figure(figsize=(8, 6))
    sns.heatmap(confusion_matrix(y_test, y_pred), annot=True, fmt='d', cmap='Blues')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.title('Confusion Matrix')
    plt.show()

In [ ]:
# Feature importance
if hasattr(pipeline['model'], 'feature_importances_'):
    # Get feature names after preprocessing (more complex with OneHotEncoder)
    feature_names = []
    for name, transformer, features in pipeline['preprocessor'].transformers_:
        if name == 'cat':
            # Get the categories from OneHotEncoder
            for i, feature in enumerate(features):
                categories = pipeline['preprocessor'].named_transformers_[name].named_steps['onehot'].categories_[i]
                for category in categories:
                    feature_names.append(f"{feature}_{category}")
        else:
            feature_names.extend(features)
    
    # Plot feature importance
    importances = pipeline['model'].feature_importances_
    indices = np.argsort(importances)[-15:]  # Get top 15 features
    
    plt.figure(figsize=(12, 8))
    plt.barh(range(len(indices)), importances[indices], align='center')
    try:
        plt.yticks(range(len(indices)), [feature_names[i] for i in indices])
    except IndexError:
        # If feature names don't match, use generic names
        plt.yticks(range(len(indices)), [f"Feature {i}" for i in indices])
    plt.xlabel('Feature Importance')
    plt.title('Top 15 Important Features for Disease Prediction')
    plt.show()

## 6. Model Interpretation and Conclusions

Based on our model analysis, we can draw the following conclusions about the relationship between climate variables and disease incidence:

1. The most important climate factors affecting disease prevalence appear to be [based on feature importance].
2. The model performance metrics indicate [good/moderate/poor] predictive ability.
3. Seasonal variations show that disease risk is higher during [season based on data].
4. Temperature and humidity interactions suggest that [interpretation of interaction effects].

These findings could be valuable for public health planning and disease prevention strategies.

## 7. Next Steps and Recommendations

- Collect more granular climate data to improve prediction accuracy
- Incorporate additional variables like population density and socioeconomic factors
- Try more complex models (e.g., gradient boosting, neural networks)
- Perform time-series analysis to forecast disease outbreaks
- Validate findings with domain experts in epidemiology and climate science